## DBTC-1456
### Steps
* Re-run primary.deposit table for all businessdates

## DBTC-1449 
## Portfolio & Assignment Update

In [0]:
%sql
    UPDATE `p-cdh`.primaryactive.portfolio
    SET    portfolio_id_qca = 18
    WHERE  PortfolioName = 'MONZA PORTFOLIO - RETAIL'
    AND  portfolio_id_qca IS NULL;

In [0]:
%sql
UPDATE `p-cdh`.primaryactive.portfolio
SET    portfolio_id_qca = 19
WHERE  PortfolioName = 'MONZA PORTFOLIO - WHOLESALE'
AND  portfolio_id_qca IS NULL;

In [0]:
%sql
UPDATE `p-cdh`.primaryactive.assignment
SET    PortfolioKey  = 18,
       PortfolioName = 'MONZA PORTFOLIO - RETAIL'
WHERE  PortfolioKey IS NULL
AND  BusinessDate <= '2025-03-13'
AND  substring(cdi,1,3) = 'MZR';

In [0]:
%sql
UPDATE `p-cdh`.primaryactive.assignment
SET    PortfolioKey  = 19,
       PortfolioName = 'MONZA PORTFOLIO - WHOLESALE'
WHERE  PortfolioKey IS NULL
AND  BusinessDate <= '2025-03-13'
AND  substring(cdi,1,3) = 'MZW';

## DBTC-1473
## Lg Archive Update

In [0]:
%sql
CREATE OR REPLACE TABLE `p-cdh`.primaryarchive.lg
AS
SELECT * FROM `qa-cdh`.reference.lg_hist_test

## DBTC-1477
## Interaction Update

In [0]:
%sql
    UPDATE `p-cdh`.primaryactive.interaction
    SET    PortfolioKey  = 25,
       PortfolioName = 'EUCLID'
    WHERE  PortfolioKey IS NULL
      AND  substring(cdi, 1, 3) = 'EUC';

In [0]:
%sql
UPDATE `p-cdh`.primaryactive.interaction
SET    PortfolioKey  = 20,
       PortfolioName = 'DOMUS PORTFOLIO - RETAIL'
WHERE  PortfolioKey IS NULL
  AND  substring(cdi, 1, 3) = 'DMR';

In [0]:
%sql
UPDATE `p-cdh`.primaryactive.interaction
SET    PortfolioKey  = 21,
       PortfolioName = 'DOMUS PORTFOLIO - WHOLESALE'
WHERE  PortfolioKey IS NULL
  AND  substring(cdi, 1, 3) = 'DMW';

In [0]:
%sql
UPDATE `p-cdh`.primaryactive.interaction
SET    PortfolioKey  = 22,
       PortfolioName = 'RHODIUM PORTFOLIO - RETAIL'
WHERE  PortfolioKey IS NULL
  AND  substring(cdi, 1, 3) = 'RDR';

In [0]:
%sql
UPDATE `p-cdh`.primaryactive.interaction
SET    PortfolioKey  = 23,
       PortfolioName = 'RHODIUM PORTFOLIO - WHOLESALE'
WHERE  PortfolioKey IS NULL
  AND  substring(cdi, 1, 3) = 'RDW';

In [0]:
%sql
UPDATE `p-cdh`.primaryactive.interaction
SET    PortfolioKey  = 19,
       PortfolioName = 'MONZA PORTFOLIO - WHOLESALE'
WHERE  PortfolioKey IS NULL
  AND  substring(cdi, 1, 3) = 'MZW';

In [0]:
%sql
UPDATE `p-cdh`.primaryactive.interaction
SET    PortfolioKey  = 18,
       PortfolioName = 'MONZA PORTFOLIO - RETAIL'
WHERE  PortfolioKey IS NULL
  AND  substring(cdi, 1, 3) = 'MZR';

## ITL-117

## SettlementExposure Active Update

In [0]:
%sql
ALTER TABLE `p-cdh`.primaryactive.settlementexposure
ADD COLUMNS AgreementKey STRING after ExposureKey


In [0]:
%sql
ALTER TABLE `p-cdh`.primaryactive.settlementexposure
ADD COLUMNS CancellationReasonCode INT after LastUpdatedDate

In [0]:
%sql
ALTER TABLE `p-cdh`.primaryactive.settlementexposure
ADD COLUMNS CancellationReason STRING after CancellationReasonCode

In [0]:
%sql
ALTER TABLE `p-cdh`.primaryactive.settlementexposure
ADD COLUMNS DiasRFCode STRING after CancellationReason

## SettlementExposure Archive Update

In [0]:
%sql
    ALTER TABLE `p-cdh`.primaryarchive.settlementexposure
    ADD COLUMNS AgreementKey STRING after ExposureKey

In [0]:
%sql
    ALTER TABLE `p-cdh`.primaryarchive.settlementexposure
    ADD COLUMNS CancellationReasonCode INT after LastUpdatedDate

In [0]:
%sql
    ALTER TABLE `p-cdh`.primaryarchive.settlementexposure
    ADD COLUMNS CancellationReason STRING after CancellationReasonCode

In [0]:
%sql
    ALTER TABLE `p-cdh`.primaryarchive.settlementexposure
    ADD COLUMNS DiasRFCode STRING after CancellationReason

## PaymentSchedule Active Update

In [0]:
%sql
    ALTER TABLE `p-cdh`.primaryactive.paymentschedule
    ADD COLUMNS PartyKey STRING after ExposureKey

In [0]:
%sql
ALTER TABLE `p-cdh`.primaryactive.paymentschedule
ADD COLUMNS AgreementKey STRING after PartyKey 

In [0]:
%sql
    ALTER TABLE `p-cdh`.primaryactive.paymentschedule
    ADD COLUMNS PayTolerance int after SettlementCaseRecordKey

In [0]:
%sql
    ALTER TABLE `p-cdh`.primaryactive.paymentschedule
    ADD COLUMNS PayPercent decimal(12,6) after PayTolerance

In [0]:
%sql
    ALTER TABLE `p-cdh`.primaryactive.paymentschedule
    ADD COLUMNS PayPercentAmount decimal(18,2) after PayPercent

## Refresh views

In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('p-cdh')
      AND table_schema in ('derivative', 'primary')
      and table_type='VIEW'
""")

# display(views_df)

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    # print(table_name)

    full_view_name =f"`{table_catalog}`.{table_schema}.{table_name}"

    describe_full_view_name = spark.sql(f"""
              DESCRIBE EXTENDED {full_view_name}""")
    
    describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

   # print(describe_full_view_name)

    describe_query = spark.sql(f"""
              select data_type--REGEXP_REPLACE(TRIM(data_type), '[^a-zA-Z0-9*<> ]', '') 
              AS cleaned_view_text 
              from describe_full_view_name 
              where col_name ='View Text'
              """)
    
    sql_view_creation = describe_query.collect()[0][0]

    create_or_replace_sql = f"""
          create or replace view {full_view_name} as 
          {sql_view_creation}
          """
    
    spark.sql(create_or_replace_sql)



## DBTC-1324
Closed loans function update

In [0]:
# Get the schema of the dexposure table
dexposure_table = spark.table(f"`p-cdh`.derivative.dexposure")
dexposure_schema_fields = dexposure_table.schema.fields


# Convert the schema fields into the required string format
dexposure_schema = ",\n".join([
    f"{field.name} {field.dataType.simpleString()}"
    for field in dexposure_schema_fields
])


spark.sql(f'''
CREATE OR REPLACE FUNCTION `p-cdh`.derivative.get_all_exposures_by_business_date(
    business_date Date
)
RETURNS TABLE 
( 
{dexposure_schema}
)
RETURN

SELECT B.*
FROM(
        SELECT  ExposureKey, MAX(BusinessDate) BusinessDate
        FROM `p-cdh`.derivative.dexposure
        WHERE BusinessDate <= business_date
        GROUP BY ExposureKey
) A
INNER JOIN   `p-cdh`.derivative.dexposure B
on a.ExposureKey = b.ExposureKey
and a.BusinessDate = b.BusinessDate
AND B.IsActiveFlagInt = 0
AND coalesce(B.JustificationCode,0) != 43023

UNION

SELECT * FROM `p-cdh`.derivative.dexposure
WHERE BusinessDate = business_date
AND IsActiveFlagInt = 1
''')

## DBTC-1478
Closed loans function update for Exposure

In [0]:
# Get the schema of the exposure table
exposure_table = spark.table(f"`p-cdh`.primary.exposure")
exposure_schema_fields = exposure_table.schema.fields


# Convert the schema fields into the required string format
exposure_schema = ",\n".join([
    f"{field.name} {field.dataType.simpleString()}"
    for field in exposure_schema_fields
])


spark.sql(f'''
CREATE OR REPLACE FUNCTION `p-cdh`.primary.get_all_exposures(
    business_date Date
)
RETURNS TABLE 
( 
{exposure_schema}
)
RETURN

SELECT B.*
FROM(
        SELECT  ExposureKey, MAX(BusinessDate) BusinessDate
        FROM `p-cdh`.primary.exposure
        WHERE BusinessDate <= business_date
        GROUP BY ExposureKey
) A
INNER JOIN   `p-cdh`.primary.exposure B
on a.ExposureKey = b.ExposureKey
and a.BusinessDate = b.BusinessDate
AND B.IsActiveFlag = false
AND coalesce(B.JustificationCode,0) != 43023

UNION

SELECT * FROM `p-cdh`.primary.exposure
WHERE BusinessDate = business_date
AND IsActiveFlag = true
''')